## STEP 1 Knowledge Graph 기초

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd

load_dotenv()
client = OpenAI()
df = pd.read_excel("Articles_20260309_192845.xlsx")

text = df.iloc[0]['content']

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": """뉴스 기사에서 (주어, 관계, 목적어) 형태의 트리플을 추출하세요.  각 줄에 하나씩, 형식: (주어, 관계, 목적어)
  최대 10개만 추출하세요."""},
        {"role": "user", "content": text}
    ]
)

print("=== 추출된 트리플 ===")
print(response.choices[0].message.content)

In [ ]:
all_triples = []

for idx in range(5):  # 기사 5개만
    text = df.iloc[idx]['content']
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": """뉴스 기사에서 (주어, 관계, 목적어) 트리플을 추출하세요.
  각 줄에: (주어, 관계, 목적어)
  최대 5개만."""},
            {"role": "user", "content": text}
        ]
    )

    triples_text = response.choices[0].message.content
    print(f"\n--- 기사 {idx}: {df.iloc[idx]['title'][:30]} ---")
    print(triples_text)

    # 파싱
    for line in triples_text.strip().split("\n"):
        line = line.strip().strip("()")
        parts = [p.strip().strip("()") for p in line.split(",")]
        if len(parts) == 3:
            all_triples.append(tuple(parts))

print(f"\n총 트리플 수: {len(all_triples)}")

In [ ]:
from collections import Counter

# 모든 주어와 목적어 수집
entities = []
for s, r, o in all_triples:
    entities.append(s)
    entities.append(o)

counter = Counter(entities)
print("=== 자주 등장하는 엔티티 ===")
for entity, count in counter.most_common(10):
    print(f"  {entity}: {count}회")

## STEP 2 Neo4j & Cypher 쿼리

```shell
docker run -d \
  --name neo4j \
  -p 7474:7474 -p 7687:7687 \
  -e NEO4J_AUTH=neo4j/password123 \
  neo4j:latest
```

In [ ]:
# uv add neo4j
from neo4j import GraphDatabase

driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password123"))

with driver.session() as session:
    result = session.run("RETURN 'Neo4j 연결 성공!' AS message")
    print(result.single()["message"])

In [ ]:
  with driver.session() as session:
    # 노드 생성
    session.run("CREATE (t:Person {name: $name, role: $role})", name="트럼프", role="대통령")
    session.run("CREATE (c:Country {name: $name})", name="미국")
    session.run("CREATE (c:Country {name: $name})", name="이란")

    # 관계 생성
    session.run("""
          MATCH (p:Person {name: "트럼프"}), (c:Country {name: "미국"})
          CREATE (p)-[:대통령]->(c)
      """)
    session.run("""
          MATCH (a:Country {name: "미국"}), (b:Country {name: "이란"})
          CREATE (a)-[:전쟁중]->(b)
      """)

    # 검색
    result = session.run("""
          MATCH (p:Person)-[:대통령]->(c:Country)-[:전쟁중]->(target)
          RETURN p.name AS person, target.name AS target
      """)
    for record in result:
        print(f"{record['person']}은 {record['target']}과 전쟁 중인 나라의 대통령")

## STEP 3 데이터로 그래프 만들기

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
from neo4j import GraphDatabase
import pandas as pd
import json

load_dotenv()
client = OpenAI()
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "password123"))

df = pd.read_excel("Articles_20260309_192845.xlsx")

def extract_triples(text):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": """뉴스 기사에서 엔티티와 관계를 추출하세요.
  JSON 배열로 출력. 각 항목: {"subject": "주어", "subject_type": "타입", "relation": "관계", "object": "목적어",
  "object_type": "타입"}
  타입은: Person, Organization, Country, Event, Weapon, Place 중 하나.
  최대 7개만. JSON만 출력하세요. 마크다운 코드블록 쓰지 마세요."""},
            {"role": "user", "content": text}
        ],
        response_format={"type": "json_object"}
    )
    try:
        result = json.loads(response.choices[0].message.content)
        # {"triples": [...]} 형태일 수도 있으니 처리
        if isinstance(result, dict):
            for key in result:
                if isinstance(result[key], list):
                    return result[key]
        if isinstance(result, list):
            return result
        return []
    except:
        return []

# 테스트
triples = extract_triples(df.iloc[0]['content'])
for t in triples:
    print(f"({t['subject']}) -[{t['relation']}]-> ({t['object']})")


In [ ]:
# 기존 데이터 정리
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")

all_triples = []

for idx, row in df.iterrows():
    if not isinstance(row['content'], str):
        continue

    triples = extract_triples(row['content'])
    print(f"[{idx+1}/{len(df)}] {row['title'][:30]}... → {len(triples)}개 트리플")

    for t in triples:
        t['source_article'] = row['title']
        t['category'] = row['category']
        all_triples.append(t)

print(f"\n총 트리플 수: {len(all_triples)}")

In [ ]:
import re

def clean_rel_name(name):
    """Neo4j 관계 타입에 쓸 수 있도록 정제"""
    name = name.replace(" ", "_")
    name = re.sub(r'[^a-zA-Z0-9가-힣_]', '', name)  # 허용 문자만 남김
    if name and name[0].isdigit():
        name = "R_" + name  # 숫자로 시작하면 접두사 추가
    return name or "RELATED"

with driver.session() as session:
    for t in all_triples:
        try:
            rel_type = clean_rel_name(t['relation'])
            session.run("""
                  MERGE (s {name: $subject})
                  SET s:""" + t['subject_type'] + """
                  MERGE (o {name: $object})
                  SET o:""" + t['object_type'] + """
                  MERGE (s)-[r:""" + rel_type + """]->(o)
              """, subject=t['subject'], object=t['object'])
        except Exception as e:
            print(f"오류: {e}")
            continue

print("Neo4j 저장 완료!")

In [ ]:
with driver.session() as session:
    # 전체 노드/관계 수
    nodes = session.run("MATCH (n) RETURN count(n) AS cnt").single()["cnt"]
    rels = session.run("MATCH ()-[r]->() RETURN count(r) AS cnt").single()["cnt"]
    print(f"노드: {nodes}개, 관계: {rels}개")

    # 가장 연결이 많은 엔티티 Top 10
    result = session.run("""
          MATCH (n)-[r]-()
          RETURN n.name AS name, labels(n) AS type, count(r) AS connections
          ORDER BY connections DESC
          LIMIT 10
      """)
    print("\n=== 허브 엔티티 (연결 많은 순) ===")
    for record in result:
        print(f"  {record['name']} ({record['type'][0]}): {record['connections']}개 연결")

In [ ]:
with driver.session() as session:
    # "이란과 관련된 모든 엔티티는?"
    result = session.run("""
          MATCH (n)-[r]-(target {name: "이란"})
          RETURN n.name AS entity, type(r) AS relation
      """)
    print("=== 이란과 관련된 엔티티 ===")
    for record in result:
        print(f"  {record['entity']} - {record['relation']}")

    # "트럼프에서 2홉 이내 엔티티는?"
    result = session.run("""
          MATCH (start {name: "트럼프"})-[r1]-(mid)-[r2]-(end)
          RETURN DISTINCT end.name AS entity, type(r1) AS rel1, mid.name AS via, type(r2) AS rel2
          LIMIT 10
      """)
    print("\n=== 트럼프에서 2홉 이내 ===")
    for record in result:
        print(f"  트럼프 -{record['rel1']}-> {record['via']} -{record['rel2']}-> {record['entity']}")